In [ ]:
# ---------------------------------------------------------------------------------
# (1) Loading necessary libraries.

import pandas as pd
import pathlib as path
import numpy as np
import datetime as dt
import calendar
from string import digits


digits:list[str] = list(digits)
today: dt.date = dt.date.today()



# Determine the minimum and maximum number of clients per trip.
n_client_range: tuple[int] = (25, 70)

# Set the company's age (how many years it has been operating).
company_age: int = 5

# Determine the minimum and maximum number of trips.
n_trip_range:tuple[int] = (15*company_age, 25*company_age)


# Calculate the number of clients to generate based on the above parameters.
n_clients:int = (n_trip_range[1] + n_trip_range[0])//2 * n_client_range[1]

n_female_clients: int = np.random.randint(1, n_clients+1)
n_male_clients:int = n_clients - n_female_clients

# For how many people there is one driver and one guide.
driver_ratio: int = 50
guide_ratio: int = 25


# ---------------------------------------------------------------------------------
# (2) Reading popular male and female first and last names.

# Find the directory with names.
name_dir_path: path.Path = path.Path().cwd()/"Names"

# Find the directory where we save all .csv tables.
csv_tables_dir_path: path.Path = path.Path().cwd()/"CSV_Tables"


def read_names(first_names_file:str, last_names_file:str, dir_path:path.Path) -> tuple[pd.Series, pd.Series]:
    """This function reads files containing the most popular Polish first and last names. It loads them into a pandas Series."""
    # Read first and last names.
    first_names: pd.DataFrame = pd.read_excel(dir_path/first_names_file  ,"Arkusz1",
                                                usecols = [0,2]
                                                )
    last_names: pd.DataFrame = pd.read_excel(dir_path/last_names_file, "Arkusz1",
                                                nrows = 5_000) # There are many rows in the sheet (over 380,000).
                                                                # Read only a certain part of the rows.
                                                                

    # We only consider the most popular first and last names.
    first_names = first_names.loc[first_names["LICZBA_WYSTĄPIEŃ"] > 500, "IMIĘ_PIERWSZE"]
    last_names = last_names.loc[last_names["Liczba"] > 500, "Nazwisko aktualne"] 

    # Shorten column names.
    first_names.columns = ['first_name']
    last_names.columns = ['last_name']

    return first_names, last_names

# Load male first and last names.
male_first_names, male_last_names = read_names("imiona_meskie.xlsx", "nazwiska_meskie.xlsx", name_dir_path)

# Load female first and last names.
female_first_names, female_last_names = read_names("imiona_zenskie.xlsx", "nazwiska_zenskie.xlsx", name_dir_path)



# ---------------------------------------------------------------------------------
# (3) Helper generator functions.

def generate_birthday() -> dt.datetime:
    # Generate age from a normal distribution. Remember that the person must be at least an adult.
    birth_year: int =  today.year - max(int(np.random.normal(42.9, 15)), 18)



    birth_month:int = np.random.randint(1, 13) # Generate birth month.

    birth_day = calendar.monthrange(birth_year, birth_month)[1] # Find the day of birth (consistent with the number of days in the month)

    birthday = dt.date(birth_year, birth_month, birth_day) # Combine all birthday components into one date.

    return birthday



def generate_email(first_name:str, last_name:str, domains:list[str]) -> str:
    email:str = first_name.lower() + last_name.lower() + "".join(np.random.choice(digits, 3)) +"@"+ np.random.choice(domains)

    return email



def generate_phone_number() -> str:
    # Generate a random phone number that does not start with 0.
    phone_number:np.array = np.random.choice(digits, 9)

    # If we drew a zero, replace the digit.
    if phone_number[0] == "0":
        phone_number[0] = np.random.choice(digits[1:])

    phone_number: str = "".join(phone_number)

    return phone_number


def insert_id_col(people_data:pd.DataFrame, col_name:str) -> pd.DataFrame:
    people_data.insert(loc = 0,column = col_name,
                       value = range(1, people_data.shape[0]+1))
    
    return people_data


def merge_females_and_males(female_data:pd.DataFrame, male_data:pd.DataFrame, id_col_name:str) ->pd.DataFrame:
    people_data_df : pd.DataFrame = pd.concat(objs = [female_data, male_data], axis = 0).reset_index(drop = True)

    people_data_df : pd.DataFrame = insert_id_col(people_data_df, id_col_name)

    return people_data_df


def chance_of_being_student(age:float):
    """
    Calculates the chance of being a student based on age.

    Parameters:
        age (int): The person's age.

    Returns:
        float: The chance of being a student (from 0.0 to 1.0).
    """
    if age < 20 or age > 35:
        return 0.0
    else:
        return 1 - (age - 20) / 15  # Linear decrease in chance



# ---------------------------------------------------------------------------------
# (4) Creating client personal data.

def generate_client_personal_data(first_names:pd.Series, last_names:pd.Series, n:int) -> pd.DataFrame:
    """This function generates personal data for clients. It creates rows for first name, last name, date of birth, phone number, email address, and student status.

    """
    # Prepare the output dataframe.
    clients_df: pd.DataFrame = pd.DataFrame(data = {"first_name":pd.Series(dtype = pd.StringDtype()),
                                                    "last_name": pd.Series(dtype = pd.StringDtype()), 
                                                    "birth_date":pd.Series(dtype = "datetime64[ns]"),
                                                    "discount": pd.Series(dtype = pd.Float32Dtype()),
                                                    "phone_number":pd.Series(dtype = pd.StringDtype()), 
                                                    "email_address": pd.Series(dtype = pd.StringDtype()),
                                                    "student_status":pd.Series(dtype = pd.Int8Dtype())}                                          
    )

    clients_df["birth_date"] = clients_df['birth_date'].dt.date

    # List of email domains.
    email_domains:list[str] = ['gmail.com', "wp.pl", "onet.pl"]


    for id in range(n):
        first_name_idx:int = np.random.randint(0, first_names.shape[0])
        last_name_idx:int = np.random.randint(0, last_names.shape[0])

        first_name:str = first_names[first_name_idx].capitalize() # Draw a first name
        last_name:str = last_names[last_name_idx].capitalize()  # Draw a last name.

        
        # Generate a random phone number that does not start with 0.
        phone_number:str = generate_phone_number()

        # Create an email for this client
        email:str = generate_email(first_name, last_name, email_domains)
        
        
        # Make up a birth date for this client. Everyone must be an adult and we assume they live a maximum of 100 years.
        birthday = generate_birthday()
        
        age: int = (today-birthday).days/365

        # Is a student
        is_student: int = np.random.binomial(1, chance_of_being_student(age))
     
        # Determine the discount based on age.
        if is_student:
            discount = 0.49
        else:
            if age >= 70:
                discount = 0.5
            elif age >= 60:
                discount = 0.65
            elif age >= 50:
                discount = 0.75
            elif age >= 40:
                discount = 0.85
            elif age >= 30:
                discount = 0.9
            else:
                discount = 1
    

        clients_df.loc[id, :] = [first_name, last_name, birthday, discount,phone_number, email, is_student]

    return clients_df

        

female_clients_df : pd.DataFrame = generate_client_personal_data(female_first_names, female_last_names, n_female_clients)
male_clients_df : pd.DataFrame = generate_client_personal_data(male_first_names, male_last_names, n_male_clients)

clients_df = merge_females_and_males(female_clients_df, male_clients_df, "client_id")


clients_df.to_csv(path_or_buf = csv_tables_dir_path/"Clients_db.csv",
                    sep = ";", index = False)




# ---------------------------------------------------------------------------------
# (5) Creating the salary table.

def create_salary_table():

    salary_df:  pd.DataFrame = pd.DataFrame(data = {"position":pd.Series(dtype =pd.StringDtype()),
                                                    "salary":pd.Series(dtype = pd.Float64Dtype())
                                                    })
    
    salary_df.loc[0, :] = ["guide", 1200]
    salary_df.loc[1, :] = ["driver",800]


    salary_df.to_csv(csv_tables_dir_path/"Salary_db.csv",
                      sep = ';',
                      encoding = "utf-8",
                      index = False)
    

create_salary_table()



# ---------------------------------------------------------------------------------
# (6) Generating the planned trips table and transactions table.

def generate_startday() -> dt.datetime:
    year: int =  np.random.randint(today.year - company_age, today.year) # Draw the year of the trip.

    month:int = np.random.randint(6, 10) # Generate the month of the trip.

    day = calendar.monthrange(year, month)[1] # Find the day of birth (consistent with the number of days in the month)


    return dt.date(year, month, day)



def generate_trip_tables():
    """
    Generates tables related to trips from CSV files and processes the data.

    The function performs the following steps:
    1. Loads the "Cities_db.csv" file to get a list of possible destinations.
    2. Converts the "trip_duration" column in the cities DataFrame to timedelta type.
    3. Loads the "Clients_db.csv" file to get a list of clients.
    4. Creates a DataFrame describing planned trips with columns for trip ID, start date, whether the trip took place, and city ID.
    5. Creates a DataFrame to record completed transactions with columns for transaction ID, amount, date, and other relevant information.
    Returns:
        None
    """

    
    # Load the table containing the list of possible destinations.
    cities_df: pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Cities_db.csv",
                                          sep = ";", 
                                          encoding = "utf-8",
                                          
                                          )
    
    
    # Convert the "trip_duration" column to timedelta type.
    cities_df["trip_duration"] = pd.to_timedelta(cities_df["trip_duration"])
    
    # Load the clients table.
    clients_df: pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Clients_db.csv",
                                           sep = ";",
                                           encoding = "utf-8"
                                           , parse_dates = ["birth_date"]
                                           )
   

    # Create a table describing planned trips.
    planned_trips_df: pd.DataFrame = pd.DataFrame(data = {"trip_id":pd.Series(dtype = pd.Int32Dtype()),
                                                       "start_date":pd.Series(dtype = "datetime64[ns]"),
                                                       "took_place":pd.Series(dtype = pd.BooleanDtype())
                                                       ,"city_id":pd.Series(dtype = pd.Int32Dtype())})

    # Build a table to record completed transactions.
    transaction_df:pd.DataFrame =  pd.DataFrame(data = {"transaction_id":pd.Series(dtype = pd.Int32Dtype()),
                                         "amount":pd.Series(dtype = pd.Float64Dtype()),
                                         "transaction_date":pd.Series(dtype = "datetime64[ns]"),
                                         "client_id":pd.Series(dtype = pd.Int32Dtype()),
                                         "trip_id":pd.Series(dtype = pd.Int32Dtype())
                                         })


    
    # Determine how many trips are to be planned.
    n_trips: int = np.random.randint(low = n_trip_range[0], high = n_trip_range[1])

    # Create a unique identifier for each transaction.
    transaction_id:int = 1 

    for trip_id in range(1, n_trips+1):
        # Draw the start date of the trip.
        trip_start_date:dt.date = generate_startday()

  

        # Determine the period in which clients can pay for the ticket.
        trans_period_start : dt.date = trip_start_date - dt.timedelta(30)
        trans_period_end : dt.date = trip_start_date - dt.timedelta(7)

        # Generate a date range in which clients can pay for tickets.
        trans_period: np.array[dt.date] = pd.date_range(trans_period_start, trans_period_end, freq = "1d").date
    

        # Draw the (maximum) number of travelers
        n_max_clients: int = np.random.randint(n_client_range[0],
                                            n_client_range[1]+1)
        
        
        # Counter for the actual number of clients.
        n_client_fact: int = 0
       
        # Draw the city we are going to.
        city_id: int = cities_df["city_id"].sample().values[0]
   
        # Find the record describing the drawn city.
        city_record:pd.DataFrame = cities_df.loc[ cities_df["city_id"] == city_id, ["city_id", "trip_price","trip_duration"]]

        # Duration of the trip.
        trip_duration:pd.Timedelta = pd.to_timedelta(city_record["trip_duration"].values[0])

        # Find the end date of the trip.
        trip_end_date:dt.date = trip_start_date + trip_duration
    
        # Find the ticket price corresponding to this city.
        ticket_price: float = city_record["trip_price"].values[0]

        
        # Find the appropriate number of different clients.
        clients_ids:pd.Series = clients_df["client_id"].sample(n_max_clients, replace = False)


        # Consider adding each client individually.
        for client_id in clients_ids:
            # Find the record describing the selected client.
            client_record: pd.DataFrame = clients_df.loc[clients_df["client_id"] == client_id, :]

           
    
            # Find all transactions related to this client.
            client_transaction:pd.DataFrame = transaction_df.loc[transaction_df["client_id"] == client_id, :]

            # Find the discount applicable to this client.
            discount: float = client_record["discount"].values[0]

            # Calculate the ticket price after the discount.
            discounted_ticket: float = round(discount * ticket_price, 2)

            # Draw the transaction date.
            transaction_date: dt.date = pd.to_datetime(np.random.choice(trans_period))



            # If the client has not made any transactions yet, add them to the table immediately.
            if client_transaction.shape[0] == 0:
                transaction_df.loc[transaction_id, :] = [transaction_id, discounted_ticket,
                                                            transaction_date, client_id, trip_id]
                
                transaction_id +=1
                n_client_fact += 1

            else:
                  # Otherwise, find the trips they participated in.

                # Find the IDs of the trips the client participated in.
                client_trip_ids: pd.DataFrame = client_transaction["trip_id"]

                # Find the trips the client participated in.
                client_trips:pd.DataFrame = planned_trips_df.merge(right = client_trip_ids, on = "trip_id")

                # Now find the trips that took place.
                client_taken_trips : pd.DataFrame = client_trips.loc[client_trips["took_place"] == True, :]


                 # If the client has not yet participated in any trip, add them to the table immediately.
                if client_taken_trips.shape[0] == 0:
                    transaction_df.loc[transaction_id, :] = [transaction_id, discounted_ticket,
                                                            transaction_date, client_id, trip_id]
                    
                    transaction_id +=1
                    n_client_fact += 1
                
                else:
                    # Find the start and end dates of the trips the client participated in.
                    client_taken_trips_with_time: pd.DataFrame =client_taken_trips.merge(cities_df, on = "city_id")[["start_date", "trip_duration"]]

                    # Add a column indicating the end date of the trip.
                    client_taken_trips_with_time["end_date"] = pd.to_datetime(client_taken_trips_with_time["start_date"] +client_taken_trips_with_time["trip_duration"])
    
                    # Find the end date of the earliest trip.
                    earliest_trip_end_date = pd.to_datetime(client_taken_trips_with_time["end_date"].max()).date()

                    # Find the start date of the latest trip.
                    latest_trip_start_date = pd.to_datetime(client_taken_trips_with_time["start_date"].min()).date()
                  
                    # Add the client if the difference between the start date of this trip and the end date is at least 3.
                    if (trip_start_date - earliest_trip_end_date).days >= 3 or (latest_trip_start_date - trip_end_date).days >= 3:
                            transaction_df.loc[transaction_id, :] = [transaction_id, discounted_ticket,
                                                            transaction_date, client_id, trip_id]
                            
                            transaction_id += 1
                            n_client_fact += 1
                        
                                   
        # Determine if the trip took place based on the number of clients.
        if n_client_fact >= (3*n_client_range[0] + n_client_range[1])/4:
            took_place: bool = True
        else:
            took_place: bool = False

        # Convert the trip start date to the appropriate type so it can be saved to the table.
        trip_start_date = pd.to_datetime(trip_start_date)
        planned_trips_df.loc[trip_id, :] = (trip_id, trip_start_date, took_place, city_id)


    # Save the planned trips table to a csv file.
    planned_trips_df.to_csv(csv_tables_dir_path/"Trips_db.csv",
                           sep = ";", encoding = "utf-8", 
                           index = False)
    
    # In the 'clients' table, only include clients who have made a transaction.

    clients_df:pd.DataFrame = clients_df.loc[clients_df["client_id"].isin(transaction_df["client_id"]), ].rename(columns = {"client_id":"old_id"}).reset_index(drop = True)
    clients_df.insert(0, "client_id", pd.Series(range(1, clients_df.shape[0]+1)))

 
    # Replace old identifiers with new identifiers.
    transaction_df = transaction_df.replace( to_replace= {"client_id":clients_df["old_id"].to_list()}, 
                                            value = {"client_id": clients_df["client_id"].to_list()})

    # Drop the "old_id" column
    clients_df.drop(columns = ["old_id"], inplace = True)


    # Save the transactions table to a csv file.
    transaction_df.to_csv(csv_tables_dir_path/"Transactions_db.csv",
                           sep = ";", encoding = "utf-8", 
                           index = False)
    

    # Save the clients table to a csv file.
    clients_df.to_csv(csv_tables_dir_path/"Clients_db.csv",
                           sep = ";", encoding = "utf-8", 
                           index = False)







    

generate_trip_tables()



# ---------------------------------------------------------------------------------
# (7) Generating close relatives.

def generate_one_relative(first_names:pd.Series, last_names:pd.Series, id:int):
    """This function generates one close person for a given client"""
    first_name:str = np.random.choice(first_names)
    last_name:str = np.random.choice(last_names)

    phone_number: str = generate_phone_number()
    email = generate_email(first_name, last_name, ["gmail.com", "wp.pl", "onet.pl"])

    return (id, first_name, last_name, phone_number, email)



def generate_relatives():
    def generate_noncolisionary_relative(id: int, first_names:pd.Series, last_names:pd.Series,
                                         client_phone:str) -> tuple:
        # A function that generates a relative for a given client whose phone number is different from the client's number.
        # Generate until we assign a different phone number.
        relative_record:tuple = generate_one_relative(first_names, last_names, id)

        while relative_record[3] == client_phone:
            relative_record:tuple = generate_one_relative(first_names, last_names, id)

        return relative_record
    

    clients:pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Clients_db.csv",
                          sep = ";",
                          encoding = "utf-8", usecols=["client_id", "phone_number"])
    
    clients_id: pd.Series = clients["client_id"]
    
    clients.set_index("client_id", inplace = True)



    relationship_df : pd.DataFrame = pd.DataFrame(data = {"client_id":pd.Series(dtype = pd.Int16Dtype()),
                                                          "relative_id":pd.Series(dtype = pd.Int16Dtype())})
    
    relatives_df: pd.DataFrame = pd.DataFrame(data = {"relative_id":pd.Series(dtype = pd.Int16Dtype()),
                                                        "first_name":pd.Series(dtype = pd.StringDtype()),
                                                        "last_name": pd.Series(dtype = pd.StringDtype()), 
                                                        "phone_number":pd.Series(dtype = pd.StringDtype()), 
                                                        "email_address": pd.Series(dtype = pd.StringDtype())}) # id of the client with whom the person is related.


    how_many_relatives:pd.DataFrame = pd.DataFrame(data = {"client_id":clients_id,
                                                        "n_relatives":np.zeros(clients_id.shape[0])}
                                                        )


    id: int = 1 # Relative's identifier.
    i: int  = 0 # Row identifier in the relationship_df frame.


    for client_id in clients_id:
        n_relatives = np.random.randint(1, 4) # Generate how many relatives a person can have.

        for _ in range(n_relatives):
            gender_draw:float = np.random.uniform(0, 1)

            # Depending on the person's gender, find such lists of names.
            first_names:pd.Series = female_first_names if gender_draw > 0.5 else male_first_names
            last_names:pd.Series = female_last_names if gender_draw > 0.5 else male_last_names

            # Find the client's phone number.
            client_phone: str = clients.loc[client_id, "phone_number"]

            relative:tuple = generate_noncolisionary_relative(id, first_names, last_names, client_phone)

            # Add a new close person to the 'Relatives' table
            relatives_df.loc[id, :] = relative

            # Connect 'relative' and 'client_id' with a close relationship
            relationship_df.loc[i, :] = (client_id, id)
            
            how_many_relatives.loc[client_id-1, "n_relatives"] += 1

            i += 1

            # Now generate a few other clients with whom this close person will be in a relationship
        
            # Find potential clients with whom the established close person can become friends.
            potential_players_ids = (how_many_relatives["client_id"]!=client_id) & (how_many_relatives["n_relatives"] < 4)
            potential_players = how_many_relatives.loc[potential_players_ids, :]
            
            # Determine with how many people a given person will be in a close relationship.
            n_clients:int = min(np.random.randint(0, 3), potential_players_ids.shape[0])

            for other_client_id in potential_players['client_id'].sample(n_clients):
                if clients_df.loc[other_client_id, "phone_number"] != relative[3]:
                    relationship_df.loc[i, :] = (other_client_id, id)
                    
                    i += 1

            id +=1


    # Save the frame with relatives to csv.
    relatives_df.to_csv(csv_tables_dir_path/"Relatives_db.csv",
                      sep = ';',
                      encoding = "utf-8",
                      index = False)
    
    # Save the frame with relationships to csv.
    relationship_df.to_csv(csv_tables_dir_path/"Relationships_db.csv",
                      sep = ';',
                      encoding = "utf-8",
                      index = False)
    

    
generate_relatives()


# ---------------------------------------------------------------------------------
## (8) Creating employee personal data.
def generate_worker_personal_data(first_names:pd.Series, last_names:pd.Series, trades_n:dict[str,int], email_domains:str) -> pd.DataFrame:
    """This function generates personal data for employees. It creates rows for first name, last name, phone number, and company email address.

    """
    # Prepare the output dataframe.
    workers_df: pd.DataFrame = pd.DataFrame(data = {"first_name":pd.Series(dtype = pd.StringDtype()),
                                                    "last_name": pd.Series(dtype = pd.StringDtype()), 
                                                    "position":pd.Series(dtype = pd.CategoricalDtype(categories = ["driver","guide"],
                                                                                                  ordered = False
                                                                                                  )),
                                                    "phone_number":pd.Series(dtype = pd.StringDtype()), 
                                                    "email_address": pd.Series(dtype = pd.StringDtype())},                                           
    )

    id: int = 0
    for pos in trades_n:
        for _ in range(trades_n[pos]):
            first_name_idx:int = np.random.randint(0, first_names.shape[0])
            last_name_idx:int = np.random.randint(0, last_names.shape[0])

            first_name:str = first_names[first_name_idx].capitalize() # Draw a first name
            last_name:str = last_names[last_name_idx].capitalize()  # Draw a last name.

            # Generate a random phone number that does not start with 0.
            phone_number:np.array = np.random.choice(digits, 9)

            # If we drew a zero, replace the digit.
            if phone_number[0] == "0":
                phone_number[0] = np.random.choice(digits[1:], 1)[0]

            phone_number: str = "".join(phone_number)

            # Create an email for this employee
            random_domain = np.random.choice(email_domains)
            email:str = first_name.lower() + last_name.lower() + "".join(np.random.choice(digits, 3)) +"@" + random_domain
            

            workers_df.loc[id, :] = [first_name, last_name, pos ,phone_number, email]

            id += 1

    return workers_df


def generate_workers():
    # Count how many clients have made transactions.
    n_clients:int = pd.read_csv(csv_tables_dir_path/"Transactions_db.csv",
                            sep = ";", 
                            usecols = [0]).iloc[:, 0].unique().shape[0]
    
    

    # Twice as many guides as drivers..
    female_drivers : int = int(np.ceil(n_clients/driver_ratio))
    females_guides: int = 2*female_drivers

    male_drivers:int = int(np.ceil(n_clients/driver_ratio))
    male_guides = 2*male_drivers

    female_trades_n: dict[str, int] = {"guide":females_guides, 
                                    "driver":female_drivers}

    male_trades_n: dict[str, int] = {"guide":male_guides, 
                                    "driver":male_guides}


    # Generate personal data for female employees
    email_domains:list[str] = ["nowowiejska.com"]

    female_workers_df : pd.DataFrame = generate_worker_personal_data(female_first_names, female_last_names, female_trades_n, email_domains)

    male_workers_df : pd.DataFrame = generate_worker_personal_data(male_first_names, male_last_names, male_trades_n, email_domains)

    # Combine female and male employees.
    workers_data_df : pd.DataFrame = merge_females_and_males(female_workers_df, male_workers_df, "employee_id")


    # Save employee data to a .csv file.
    workers_data_df.to_csv(path_or_buf = csv_tables_dir_path/"Employees_db.csv",
                        sep = ";", index = False)


generate_workers()



# ---------------------------------------------------------------------------------
# (9) Assigning employees.

def one_type_workers_to_trip(trips_and_workers_df:pd.DataFrame, trip_record:pd.DataFrame, trip_id:int, taken_trips_detail_df:pd.DataFrame,
                             position_df:pd.DataFrame, n_workers:int, employ_id:int):
        i: int = 0
        while i < n_workers:
            # Draw one employee.
            worker_id:int = position_df.sample(1).values[0]

            # Find records related to their participation in trips.
            worked_trips = trips_and_workers_df.loc [ trips_and_workers_df["employee_id"] == worker_id, :]

            # If this table is empty, add the employee to the table immediately.
            if worked_trips.shape[0] == 0:
                trips_and_workers_df.loc[employ_id, :] = (worker_id, trip_id)

                employ_id += 1
                i += 1

            else:
                # Otherwise, find details about the trips this employee participated in.
                worked_trips_details: pd.DataFrame = taken_trips_detail_df.loc[worked_trips["trip_id"], :]

                # Find the end date of the last trip.
                earliest_end_date:dt.date = worked_trips_details["end_date"].max()
                
                # Find the start date of the first trip.
                latest_start_date:dt.date = worked_trips_details["start_date"].min()

                # Find the end and start date of the currently considered trip.
                trip_start_date:dt.date = trip_record["end_date"]
                trip_end_date:dt.date = trip_record["start_date"]


                if (trip_start_date - earliest_end_date).days > 3 or (latest_start_date -trip_end_date).days > 3:
                    trips_and_workers_df.loc[employ_id, :] = (worker_id, trip_id)

                    employ_id += 1          
                    i += 1   




def assign_workers_to_trips():
    # Load the employees table.
    workers_df: pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Employees_db.csv",
                                           sep = ";",
                                           encoding = "utf-8")


    # Load the trips table.
    trips_df: pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Trips_db.csv",
                                           sep = ";",
                                           encoding = "utf-8")
    # Format the date.
    trips_df['start_date'] = pd.to_datetime(trips_df['start_date'])

    # Load the cities table.
    cities_df: pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Cities_db.csv",
                                           sep = ";",
                                           encoding = "utf-8")
    
    # Convert the column to timedelta type.
    cities_df['trip_duration'] = pd.to_timedelta(cities_df['trip_duration'])

    
    # Merge the "cities" table with the "trips" table to get full information about the trip.
    trips_detail_df: pd.DataFrame = trips_df.merge(cities_df, on = "city_id")

    # We only recruit employees for trips that have taken place.
    taken_trips_detail_df: pd.DataFrame =   trips_detail_df.loc[trips_detail_df["took_place"] == True, 
                                                                ["trip_id", "start_date",
                                                                 "trip_duration"]].set_index(keys = "trip_id")
    
    taken_trips_detail_df["end_date"] = taken_trips_detail_df['start_date'] + taken_trips_detail_df["trip_duration"]
   

    # Load the transactions table to count how many people are participating in each trip.
    transactions_df: pd.DataFrame = pd.read_csv(csv_tables_dir_path/"Transactions_db.csv",
                                           sep = ";",
                                           encoding = "utf-8")
    
    # Find the number of clients participating in each trip.
    n_clients: pd.Series = transactions_df["trip_id"].value_counts(sort = False)

    trips_and_workers_df: pd.DataFrame = pd.DataFrame(data = {"employee_id":pd.Series(dtype = pd.Int32Dtype()),
                                                              "trip_id":pd.Series(dtype = pd.Int32Dtype())},
                                                              )

    # Table with all guides.
    guides_df:pd.Series = workers_df.loc[workers_df["position"] == "guide", "employee_id"]
    # Table with all drivers.
    drivers_df:pd.Series = workers_df.loc[workers_df["position"] == "driver", "employee_id"]

 
    employ_id: int = 1 # Row identifier in the "trips_and_workers_df" frame

    for trip_id in taken_trips_detail_df.index:
        trip_record:pd.DataFrame = taken_trips_detail_df.loc[trip_id,:]

        n_guides:int = int(np.ceil(n_clients[trip_id]/guide_ratio))
        n_drivers: int = int(np.ceil(n_clients[trip_id]/driver_ratio))

        # Place the frames of specific positions in a list.
        workers_dfs:list[pd.DataFrame] = [guides_df, drivers_df]
        n_workers:list[int] = [n_guides, n_drivers]

        for i in range(len(workers_dfs)):
            # Add employees of a specific type.
            one_type_workers_to_trip(trips_and_workers_df, trip_record, trip_id, taken_trips_detail_df,
                                    workers_dfs[i], n_workers[i],  employ_id)
            
            employ_id += n_workers[i]


    trips_and_workers_df.to_csv(csv_tables_dir_path/"Trips_employees_db.csv",
                                sep = ";", index = False
                                )
    

assign_workers_to_trips()